# 07 · Testing the twin, building an agent, and the capstone

**Goal:** test the twin as an *interventional* model, build a tool-using agent loop on top of it
and evaluate it, then take on the open problem: a hybrid that's accurate **and** gets insulin's
sign right.

**Time:** ~1 h + the capstone · **You'll write:** `insulin_effect`, `grounded_fraction`,
`run_agent`, and then your own model

**How the exercises work.** Each one is a function with `raise NotImplementedError`. Replace that
with your code, then run the cell below it. `use(...)` checks your function against an independent
reference (the project's own code, autograd, or a closed form):
- `[ok]` → your version is used from then on.
- `[todo]` / `[fail]` → the notebook falls back to the reference solution so you can keep going.

Hints are folded under each exercise. Solutions are in `tutorials/solutions.py`; try first.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "tutorials":
    os.chdir("..")
sys.path.insert(0, "tutorials")

import numpy as np
import torch
import matplotlib.pyplot as plt
from tutor import use, close
import solutions as sol

torch.set_num_threads(4)
plt.rcParams.update({"figure.figsize": (10, 3.2), "axes.grid": True, "grid.alpha": 0.3})

import os, json, re
from biotwin.data.dataset import Cohort, standard_sets
from biotwin.models.mechanistic import MinimalModel, CorrectionNet
from biotwin.agent.tools import TwinService
from tutor import twin_report
cohort = Cohort.load("data/cohort.npz")
S = standard_sets(cohort, seed=0)
svc = TwinService(n_samples=100)
DAY = svc.list_patients()["days"][0]
print("personalised twins:", sorted(svc.post), "| first evaluation day:", DAY)

## 1. Testing an interventional model

`svc.simulate_scenario(patient, day, hour, meal_g=..., bolus_U=..., horizon_min=...)` runs the
personalised twin from the patient's recorded state at that time, twice: a baseline (basal
insulin only) and the scenario. Both runs use the **same** posterior samples, so their difference
isn't drowned by parameter noise (common random numbers). The result includes
`difference_median_mean_glucose`: the median change in mean glucose over the horizon.

We can't check a counterfactual against data; it never happened. What we **can** check are
**metamorphic relations**, properties that hold whatever the true values are. For example, more
insulin with everything else unchanged must mean less glucose.

## Exercise 1 — `insulin_effect(svc, patient, day, hour, doses)`

Return the list of `difference_median_mean_glucose` for a bolus of each dose (2-hour horizon).


<details><summary><b>Hint</b></summary>

One `simulate_scenario` call per dose, with `bolus_U=d, horizon_min=120`.

</details>

In [ ]:
def insulin_effect(svc, patient, day, hour, doses=(2, 4)):
    # TODO
    raise NotImplementedError

def check(f):
    e = f(svc, "adolescent#005", DAY, 12)
    assert len(e) == 2 and all(isinstance(x, float) for x in e), "return a list of 2 floats"
    assert e[1] < e[0] < 0, f"expected a monotone decrease for adolescent#005, got {e}"

insulin_effect = use(insulin_effect, sol.insulin_effect, check)

for p in sorted(svc.post):
    e2, e4 = insulin_effect(svc, p, DAY, 12)
    print(f"{p:15s} 2U {e2:+6.1f}  4U {e4:+6.1f}   {'ok' if e4 < e2 < 0 else 'VIOLATION'}")

One patient fails at this time: the twin says the bolus *raises* glucose. Notebook 04 showed why
(the network learned to push back against insulin). Here's a harsher test on the **population**
hybrid: every test patient, 12 start times each, 4 U bolus, deterministic model:

In [ ]:
ode = MinimalModel(); ode.load_state_dict(torch.load("results/checkpoints/ode.pt"))
hyb = MinimalModel(CorrectionNet(32)); hyb.load_state_dict(torch.load("results/checkpoints/hybrid.pt"))
print("pure ODE:      ", end=""); r_ode = twin_report(ode, cohort, S)
print("hybrid (UDE):  ", end=""); r_hyb = twin_report(hyb, cohort, S)

The physics-only model never violates the relation. The hybrid is more accurate and violates it
in a sixth of the cases, spread across several patients. **That's the trade-off the capstone asks
you to break.**

## 2. An agent on top of the twin

An LLM agent answers questions like "what happens to adult#003 if they eat 60 g at noon?" by
calling the twin as a tool. Under the hood every agent is the same small loop:

```
messages = [user question]
repeat:
    action = policy(messages)          # an LLM in practice
    if action is a final answer: return it
    result = run the requested tool    # errors are returned to the model, not raised
    append the call and its result to messages
```

## Exercise 2 — `run_agent(policy, tools, question, max_turns)`

- `policy(messages)` returns either `{"type": "final", "text": ...}` or
  `{"type": "tool_call", "name": ..., "args": {...}}`.
- `tools` maps names to Python functions.
- Return `{"answer": str, "calls": [{"name", "args", "result"}, ...]}`.
- If a tool raises, its result is `{"error": "<ExceptionType>: <message>"}`.
- After `max_turns` tool calls without a final answer, return the answer `"[max turns reached]"`.


<details><summary><b>Hint</b></summary>

Start `messages` with `{"role": "user", "content": question}`. After each tool call, append
`{"role": "assistant", "tool_call": action}` and `{"role": "tool", "name": ..., "content": result}`
so the policy can see what happened.

</details>

In [ ]:
def run_agent(policy, tools, question, max_turns=5):
    # TODO
    raise NotImplementedError

def check(f):
    def policy(msgs):
        n_tool = sum(m["role"] == "tool" for m in msgs)
        if n_tool == 0: return {"type": "tool_call", "name": "add", "args": {"a": 2, "b": 3}}
        if n_tool == 1: return {"type": "tool_call", "name": "div", "args": {"a": 1, "b": 0}}
        last = msgs[-1]["content"]
        return {"type": "final", "text": f"got {msgs[-3]['content']} then {'error' if 'error' in last else last}"}
    tools = {"add": lambda a, b: a + b, "div": lambda a, b: a / b}
    out = f(policy, tools, "q")
    assert [c["name"] for c in out["calls"]] == ["add", "div"], "wrong call sequence"
    assert out["calls"][0]["result"] == 5
    assert "ZeroDivisionError" in out["calls"][1]["result"]["error"], "tool errors should be returned as {'error': ...}"
    assert out["answer"] == "got 5 then error"
    loop = f(lambda m: {"type": "tool_call", "name": "add", "args": {"a": 1, "b": 1}}, tools, "q", max_turns=3)
    assert loop["answer"] == "[max turns reached]" and len(loop["calls"]) == 3

run_agent = use(run_agent, sol.run_agent, check)

## Exercise 3 — grounding

The agent's most important rule: **every number it states must come from a tool result.**
Write the automatic check. Find every decimal number in the answer (like `145.9` or `-2.7`).
Count it as grounded if it equals, after rounding to 1 decimal, some number anywhere in the tool
results (nested dicts and lists). Return the fraction grounded, or 1.0 if the answer has no
decimals.

Careful: in `"range 145.9-150.4"` the second number is `150.4`, not `-150.4`.


<details><summary><b>Hint</b></summary>

The regex `(?<![\d.])-?\d+\.\d+` only treats a minus sign as one when no digit comes right before
it. Collect the numbers in the results recursively; skip `bool` (it's a subclass of `int`).

</details>

In [ ]:
def grounded_fraction(answer, tool_results):
    # TODO
    raise NotImplementedError

def check(f):
    res = [{"peak": 150.43, "range": [145.9, 150.4], "nested": {"d": -2.71}}]
    close(f("peak 150.4, range 145.9-150.4, change -2.7", res), 1.0, what="all grounded")
    close(f("peak 150.4 and 199.9", res), 0.5, what="one invented")
    close(f("no numbers here", res), 1.0, what="no decimals")

grounded_fraction = use(grounded_fraction, sol.grounded_fraction, check)

Now wire it all together with a **rule-based policy** standing in for the LLM. It reads the
question with a regex, calls `simulate_scenario`, and phrases an answer from the result.

In [ ]:
def rule_policy(msgs):
    q = msgs[0]["content"].lower()
    if msgs[-1]["role"] == "tool":
        r = msgs[-1]["content"]
        if "error" in r:
            return {"type": "final", "text": f"I couldn't run that: {r['error']}"}
        s = r["with_scenario"]
        return {"type": "final", "text": f"Peak {s['peak_median']} mg/dL (vs {r['baseline']['peak_median']} without), "
                                         f"80% interval at 60 min {s['t60min_p10_p50_p90'][0]}-{s['t60min_p10_p50_p90'][2]}."}
    pm = re.search(r"(adolescent|adult|child)#(\d+)", q)
    g = re.search(r"(\d+)\s*g\b", q)
    u = re.search(r"(\d+(?:\.\d+)?)\s*u\b", q)
    return {"type": "tool_call", "name": "simulate_scenario",
            "args": {"patient": f"{pm.group(1)}#{int(pm.group(2)):03d}", "day": DAY, "hour": 12,
                     "meal_g": float(g.group(1)) if g else 0.0, "bolus_U": float(u.group(1)) if u else 0.0, "horizon_min": 120}}

tools = {"simulate_scenario": svc.simulate_scenario, "list_patients": svc.list_patients}
for q in ["What if adult#003 eats 60 g?", "adult#003 eats 60 g and takes 4 U", "What if adult#999 eats 30 g?"]:
    out = run_agent(rule_policy, tools, q)
    print(f"Q: {q}\nA: {out['answer']}\n   grounded: {grounded_fraction(out['answer'], [c['result'] for c in out['calls']]):.0%}\n")

**Swapping in an LLM.** With Claude, `policy` becomes a call to the Messages API that passes the
tool schemas (`TOOLS_SPEC` in `src/biotwin/agent/tools.py`). The model answers with `tool_use`
blocks, and you send back `tool_result` blocks. The loop has exactly the same shape as yours.
`src/biotwin/agent/agent.py` (`ClaudeAgent`) is that version, and
`scripts/04_agent_eval.py --backend claude` scores it on the 16-question golden set. The cell
below runs one question only if an API key is set.

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    from biotwin.agent.agent import ClaudeAgent
    out = ClaudeAgent(svc).ask("What happens to adult#003 if they eat seventy grams at noon?")
    print(out["answer"])
    print("grounded:", grounded_fraction(out["answer"], [c["result"] for c in out["tool_calls"]]))
else:
    print("ANTHROPIC_API_KEY not set: skipping the live Claude call.")

---
## Capstone: an accurate twin that respects physiology

**The task.** Build a hybrid whose report reads, at the same time:
- **RMSE 60 min ≤ 33.5 mg/dL** (the hybrid gets 32.5 and the pure ODE 36.8), and
- **0 / 72 insulin violations** in `twin_report`.

There's no reference solution. This is an open problem in the project. Some directions, from
least to most work:

1. **Blind the network to insulin.** Give `CorrectionNet` only $[D_1, D_2, G, \text{slope}]$.
2. **Constrain the sign.** Penalise `relu(∂f_NN/∂X)` during training (you wrote `grad_wrt_state`
   in notebook 04), or build the net so it's monotone in $X$ by construction.
3. **Shrink the network.** Add `reg_fn=lambda m: lam * (f_NN magnitude)` to `fit`. It's cheap,
   but may cost accuracy.
4. **Interventional augmentation.** Simulate extra boluses with the physics model and penalise
   the hybrid when it disagrees on the *sign* of the effect.

The scaffold below trains your model from the fitted ODE and scores it. Set `RUN = True` to train
(about `MINUTES` of CPU time). Report both numbers and what you tried. A negative result, such as
"constraint X costs 2 mg/dL", is a valid finding too.

In [ ]:
from biotwin.data.dataset import subset
from biotwin.train import fit

class MyCorrection(torch.nn.Module):
    # Start from a copy of CorrectionNet (src/biotwin/models/mechanistic.py) and change it.
    # forward(s [B,7], ctx [B]) must return dG/dt correction [B] in mg/dL/min.
    def __init__(self):
        super().__init__()
        self.net = sol.ZeroInitMLP(8, 32)
        self.scale = torch.tensor([1.0, 1.0, 1.0, 100.0, 0.05, 0.05, 1 / 100, 1.0])

    def forward(self, s, ctx):
        x = torch.cat([s, ctx[:, None]], 1) * self.scale
        x = torch.cat([x[:, :6], x[:, 6:7] - 1.4, x[:, 7:]], 1)
        return self.net(x)

RUN, MINUTES = False, 6
if RUN:
    mine = MinimalModel(MyCorrection())
    mine.log_theta.data.copy_(ode.log_theta.data)
    tr = subset(S["train"], torch.arange(len(S["train"]["y"])) % 3 == 0)
    mine, _ = fit(mine, tr, S["val"], epochs=25, lr=3e-3, bs=256, gen=torch.Generator().manual_seed(0),
                  max_minutes=MINUTES, patience=4, reg_fn=None)
    twin_report(mine, cohort, S)
else:
    print("set RUN = True to train and score your model")

## Quiz

Answer in your own words before opening the answers.

**1. Why can't we validate a counterfactual forecast against data, and what do we test instead?**

<details><summary>Answer</summary>

A counterfactual ('the same moment, but with a bolus') never happened, so there's no ground truth for it. We test invariants that must hold whatever the true values (monotonicity in dose, no intervention → no change), agreement with trusted physics on the sign of effects, and, in a simulator, the true counterfactual itself.

</details>

**2. Why do baseline and scenario share the same posterior samples in `simulate_scenario`?**

<details><summary>Answer</summary>

Common random numbers: the parameter uncertainty is mostly shared between the two runs, so it cancels in their difference. The estimate of the effect is then much less noisy than the difference of two independent samples (variance reduction through positive correlation).

</details>

**3. In the agent loop, why return tool errors to the model instead of raising?**

<details><summary>Answer</summary>

The model can recover: fix the arguments, pick a valid patient, or tell the user what went wrong. Raising would crash the conversation on recoverable mistakes. The evaluation then checks that the final answer handles the error honestly.

</details>

**4. The grounding check only looks at decimal numbers. Name one way an answer could still mislead while passing it.**

<details><summary>Answer</summary>

It could quote a real number with the wrong meaning (a baseline value presented as the scenario), state integers or words ('about two hundred'), drop the uncertainty interval or the caveats, or make a qualitative claim the tool didn't support. That's why the golden set has argument checks and refusal checks too, and why you still read the transcripts.

</details>

**5. Your capstone model gets RMSE 33.0 with 0 violations. What else would you check before claiming the fix works?**

<details><summary>Answer</summary>

Several seeds (is 33.0 vs 32.5 within noise?), more start times and doses (violations at other states?), carbs metamorphic tests (didn't break the meal response?), re-personalisation and calibration on test patients, and the per-patient breakdown. Then write down what the constraint costs.

</details>